# 04 — El grafo de punta a punta (con dobles)

**Objetivo:** recorrer el grafo completo, desde el pedido del usuario hasta el
paquete revisado, sin depender de servicios externos (LLM, OCI, vector store).

**Que vas a ver:**
1. Que el grafo se construye **inyectando dependencias**: el que cablea elige los
   servicios, el grafo no los elige.
2. Las **puertas humanas (HITL)**: el grafo se detiene y espera una respuesta.
3. El **estado final**: que claves quedan escritas y con que valores.

**Que vas a aprender:** como se verifica una pipeline de agentes sin ejecutar el
sistema real — doblando solo los servicios externos — y por que los HITL son parte
del contrato, no un detalle de interfaz.

**No necesita red, ni claves, ni cuota.** Reemplaza LLM, MCP/OCI y el checkpointer
por dobles. Los unicos caminos reales son el grafo, sus nodos y el contrato.

In [ ]:
import json
import sys
from contextlib import asynccontextmanager
from copy import deepcopy
from pathlib import Path

# El notebook corre desde notebooks/; el codigo vive en la raiz del repo.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
# `src/` tambien entra al path porque `grafo` y `llm_provider` importan de forma
# "pelada" (gotcha del AGENTS.md). Es un puente para el notebook, no el estilo.
sys.path.insert(0, str(RAIZ / "src"))

from langgraph.types import Command

from src import grafo

print("Raiz del repo:", RAIZ)

## 1. Los datos de ejemplo

Un documento corto, los chunks que el RAG habria recuperado, y la especificacion
pedagogica que produce la capa de pedagogia.

In [ ]:
DOCUMENTO_TITULO = "Guia de redes privadas en la nube"
DOCUMENTO_CONTENIDO = (
    "Una VCN es una red privada y personalizable en la nube. "
    "Las subredes organizan los recursos y las reglas controlan el trafico."
)

SOLICITUD = {
    "perfil_destinatario": "Principiante",
    "formato_salida": "Flashcards",
    "nicho_sector": "General",
    "nivel_detalle": "Didactico",
}

# Lo que el RAG habria recuperado, ya en la forma que consume el Redactor.
CHUNKS_RECUPERADOS = [
    {
        "chunk_id": "chunk-1",
        "texto": "Una VCN es una red privada y personalizable en la nube.",
        "score": 0.91,
        "pagina": 1,
        "document_id": "doc-demo",
    }
]

# Salida de la capa de pedagogia (que en produccion decide un modulo real).
ESPECIFICACION = {
    "bloom": "Entender (2)",
    "andamiaje": "Alto",
    "registro": "Cotidiano",
    "foco": "Comprension conceptual",
    "verbos": ["explicar", "identificar"],
}

print("documento:", DOCUMENTO_TITULO)
print("chunks:", len(CHUNKS_RECUPERADOS), "| especificacion:", ESPECIFICACION["foco"])

### La respuesta que daria el Redactor

El Redactor espera una salida estructurada con el contenido del paquete. Aca se
escribe a mano, con el anclaje apuntando al chunk recuperado.

In [ ]:
RESPUESTA_REDACTOR = {
    "estado": "GENERACION_CON_EVIDENCIA",
    "contenido": {
        "titulo": "Redes privadas en la nube",
        "introduccion_contextualizada": "Esta guia presenta la VCN segun el material fuente.",
        "tiempo_estimado_estudio_minutos": 5,
        "conceptos_clave": ["VCN"],
        "prerrequisitos": [],
        "items": [
            {
                "anclaje": ["chunk-1"],
                "frente": "Que es una VCN?",
                "dorso": "Una red privada y personalizable en la nube.",
                "pista_didactica": "Piensa en una red privada dentro de la nube.",
            }
        ],
    },
}

print("items:", len(RESPUESTA_REDACTOR["contenido"]["items"]))

## 2. Los dobles

Cada doble ocupa el lugar de un servicio externo y mantiene su **mismo contrato**.
El grafo no distingue si detras hay un modelo real o esto.

In [ ]:
# --- Doble del LLM estructurado que usa el Redactor ---

class ClienteEstructurado:
    """Devuelve la respuesta ya escrita, validada contra el modelo que pida el nodo."""

    def with_structured_output(self, modelo):
        self.modelo = modelo
        return self

    def invoke(self, _mensajes):
        return self.modelo.model_validate(RESPUESTA_REDACTOR)


def factory_llm(**_kwargs):
    return ClienteEstructurado()


print("doble LLM listo")

In [ ]:
# --- Doble del Supervisor: devuelve los parametros que habria extraido el LLM ---

async def supervisor_doble(_estado):
    return deepcopy(SOLICITUD)


def construir_supervisor_doble(_compartido):
    return supervisor_doble


# Sincrona a proposito: el Redactor la invoca sin `await`.
def preparar_pedagogia_doble(*_args):
    return deepcopy(ESPECIFICACION)


print("dobles de Supervisor y pedagogia listos")

In [ ]:
# --- Doble del Investigador: confirma cobertura y entrega los chunks ---

async def investigador_doble(_estado):
    return {
        "fuente_confirmada": True,
        "chunks_fuente_estructurados": deepcopy(CHUNKS_RECUPERADOS),
    }


grafo.nodo_investigador = investigador_doble

# --- Doble del Critico/Revisor: aprueba sin llamar al LLM ---

def construir_critico_doble(_compartido):
    async def critico(_estado):
        return {
            "aprobado": True,
            "evaluacion_calidad": {
                "claridad_pedagogica": "Alta",
                "observaciones": "Doble de prueba: aprueba sin LLM.",
            },
        }

    return critico


print("dobles de Investigador y Critico listos")

In [ ]:
# --- Dobles de los servicios externos: MCP/OCI y el checkpointer ---

class Tool:
    """Imita una herramienta MCP, que devuelve texto con el resultado."""

    def __init__(self, nombre):
        self.name = nombre

    async def ainvoke(self, _datos):
        if self.name == "listar_documentos_fuente":
            valor = [{"nombre": DOCUMENTO_TITULO}]
        elif self.name == "descargar_documento":
            valor = {"nombre": DOCUMENTO_TITULO, "texto_extraido": DOCUMENTO_CONTENIDO}
        else:
            valor = {"bucket": "demo", "objeto_id": "resultado.json", "status_upload": "completado"}
        return [{"type": "text", "text": json.dumps(valor)}]


@asynccontextmanager
async def conectar_mcp_doble():
    yield object()


async def tools_dobles(_sesion):
    nombres = ("listar_documentos_fuente", "descargar_documento", "guardar_resultado_formateado")
    return [Tool(nombre) for nombre in nombres]


grafo.conectar_mcp = conectar_mcp_doble
grafo.obtener_tools_langchain = tools_dobles

# El checkpointer real escribe en checkpoints.sqlite; aca va a memoria.
conectar_sqlite_real = grafo.aiosqlite.connect
CONEXIONES = []


async def sqlite_en_memoria(_ruta):
    conexion = await conectar_sqlite_real(":memory:")
    CONEXIONES.append(conexion)
    return conexion


grafo.aiosqlite.connect = sqlite_en_memoria

print("dobles de MCP y checkpointer listos")

## 3. Se construye el grafo

Todas las dependencias entran por parametro. Es la misma llamada que hara `app.py`
en produccion: alla se le pasan los servicios reales; aca, los dobles.

In [ ]:
pipeline = await grafo.construir_grafo(
    grafo.RateLimiter(),
    preparar_pedagogia=preparar_pedagogia_doble,
    get_llm_factory=factory_llm,
    construir_supervisor=construir_supervisor_doble,
    construir_critico_revisor=construir_critico_doble,
)

print("grafo compilado:", type(pipeline).__name__)

## 4. Las puertas humanas (HITL)

Al arrancar, el grafo **se detiene** y pide confirmacion antes de gastar recursos.
Se reanuda con `Command(resume=...)`. Cada parada aparece en `__interrupt__`.

In [ ]:
config = {"configurable": {"thread_id": "notebook-04"}}

estado = {
    "tema_pedido_chat": "redes VCN",
    "mensajes": [{"role": "user", "content": "Adapta la guia en flashcards para principiantes"}],
    "intentos_redactor": 0,
    "vueltas_modificacion": 0,
}

# Una respuesta por tipo de puerta; el grafo lee la que le corresponde.
RESPUESTAS = [{"confirmado": True}, {"respuesta": None}, {"instruccion": None}]

resultado = await pipeline.ainvoke(estado, config)
atendidas = 0
while resultado.get("__interrupt__") and atendidas < len(RESPUESTAS):
    print(f"--- PARADA {atendidas + 1} ---")
    print(resultado["__interrupt__"])
    resultado = await pipeline.ainvoke(Command(resume=RESPUESTAS[atendidas]), config)
    atendidas += 1

print("paradas atendidas:", atendidas)

## 5. El estado final

El grafo no devuelve un texto: deja escrito un **estado** que el resto del sistema
consume. Esto es lo que un evaluador puede inspeccionar.

In [ ]:
checkpoint = await pipeline.aget_state(config)
valores = checkpoint.values

CLAVES = (
    "documento_titulo",
    "perfil_destinatario",
    "formato_salida",
    "fuente_confirmada",
    "chunks_fuente_estructurados",
    "aprobado",
    "intentos_redactor",
    "vueltas_modificacion",
)
for clave in CLAVES:
    print(f"{clave}: {valores.get(clave)}")

print()
print("contenido_adaptado:", (valores.get("contenido_adaptado") or {}).get("titulo"))
print("metadatos:", valores.get("metadatos"))
print("evaluacion_calidad:", valores.get("evaluacion_calidad"))
print("nodos por ejecutar (vacio = termino):", checkpoint.next)

In [ ]:
# Cerrar la conexion del checkpointer (en Windows un hilo abierto cuelga el kernel).
for conexion in CONEXIONES:
    await conexion.close()

print("conexiones cerradas:", len(CONEXIONES))

## Que aprendimos

- El grafo **no elige proveedores**: recibe todo inyectado. Por eso se puede correr
  entero sin red, cambiando solo lo que se le pasa.
- Los **HITL son parte del flujo**, no de la interfaz: el grafo se detiene y espera un
  `Command(resume=...)`; sin respuesta, no avanza.
- La salida del sistema es un **estado**, no un texto: cada nodo aporta sus claves y
  los contadores quedan visibles para auditar reintentos.

**Como se extiende:** para correrlo contra los servicios reales, se reemplazan los
dobles por los reales en la llamada a `construir_grafo` — como hara `app.py`. El
notebook no cambia de forma.